## Bibliotecas

In [65]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

## Leitura do csv


In [66]:
caminho_arquivo = Path("../data/dados_anonimizados_2026.xlsx")

if caminho_arquivo.exists():
    df_original = pd.read_excel(caminho_arquivo)
else:
    print("Arquivo não encontrado.")

In [67]:
df = df_original.copy()

## Visao geral da base

Antes de analisar os numeros, conferi tamanho da base, tipos de coluna e possiveis valores faltantes.


In [68]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22946 entries, 0 to 22945
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   ano                22946 non-null  int64
 1   mes                22946 non-null  int64
 2   cidade_agrupada    22946 non-null  str  
 3   class_segmentos    22946 non-null  str  
 4   abertura_empresas  22946 non-null  int64
dtypes: int64(3), str(2)
memory usage: 896.5 KB


## Resumo numerico

Olhei um resumo das variaveis numericas para entender a faixa dos valores e identificar algum ponto estranho.

In [69]:
df.describe()

,ano,mes,abertura_empresas
count,22946.000000,22946.000000,22946.000000
mean,2021.508193,6.505230,79.910485
std,2.289839,3.449678,153.496634
min,2018.000000,1.000000,1.000000
25%,2020.000000,4.000000,12.000000
50%,2022.000000,7.000000,29.000000
75%,2024.000000,10.000000,77.000000
max,2025.000000,12.000000,2058.000000


## Maiores registros

In [70]:
df.sort_values("abertura_empresas", ascending = False).head(20)

,ano,mes,cidade_agrupada,class_segmentos,abertura_empresas
20728,2025,3,SAO PAULO,Apoio Adm,2058
19512,2024,10,SAO PAULO,Apoio Adm,1842
19681,2024,11,SAO PAULO,Apoio Adm,1801
19985,2024,12,SAO PAULO,Apoio Adm,1797
22163,2025,9,SAO PAULO,Apoio Adm,1774
17832,2024,3,SAO PAULO,Apoio Adm,1773
22346,2025,10,SAO PAULO,Apoio Adm,1767
20248,2025,1,SAO PAULO,Apoio Adm,1750
18806,2024,7,SAO PAULO,Apoio Adm,1726
19205,2024,9,SAO PAULO,Apoio Adm,1717


## Aberturas por mes

Somei as aberturas por mes para ter uma primeira leitura de sazonalidade dentro do ano.


In [ ]:
aberturas_por_mes = (
df.groupby("mes", as_index = False)
["abertura_empresas"]
.sum()
.sort_values("mes")
)

mes_maior_abertura = aberturas_por_mes.loc[aberturas_por_mes["abertura_empresas"].idxmax()]



print(aberturas_por_mes)
print("\n")
print(f"Mês que teve maior numero de abertura de empresas:{mes_maior_abertura["mes"]}")
print(f"com {mes_maior_abertura['abertura_empresas']} empresas abertas")

    mes  abertura_empresas
0     1             141640
1     2             151799
2     3             164257
3     4             146073
4     5             154213
5     6             151726
6     7             168708
7     8             167784
8     9             155460
9    10             159198
10   11             144064
11   12             128704


Mês que teve maior numero de abertura de empresas:7
com 168708 empresas abertas


## Sazonalidade media

Calculei a media mensal para ver quais meses costumam concentrar mais ou menos abertura de empresas.


In [ ]:
aberturas_ano_mes = (df.groupby( ["ano", "mes"], as_index =  False)["abertura_empresas"]
.sum()    
)

sazonalidade_mensal = (
aberturas_ano_mes.groupby( "mes",as_index = False)
["abertura_empresas"]
.mean()
.rename(columns = {"abertura_empresas" : "media_empresas_sazonal"})
.sort_values("mes")
    
)

sazonalidade_mensal

,mes,media_empresas_sazonal
0,1,17705.000
1,2,18974.875
2,3,20532.125
3,4,18259.125
4,5,19276.625
5,6,18965.750
6,7,21088.500
7,8,20973.000
8,9,19432.500
9,10,19899.750


## Periodos da pandemia

Classifiquei os anos em periodos para comparar o comportamento antes, durante e depois da pandemia.


In [71]:
def classificar_periodo(ano):
    if ano <= 2019:
        return "Pré pandemia"
    
    elif ano <= 2021:
        return "Durante pandemia"
    
    else:
        return "Pós pandemia"
    
    
df["periodo_pandemia"] = df["ano"].apply(classificar_periodo)        

## Media por periodo

Comparei a media mensal entre os periodos para entender se houve mudanca relevante no volume do mercado.


In [ ]:
abertura_mensal_periodo = (
df.groupby(["ano", "mes", "periodo_pandemia"], as_index = False)
["abertura_empresas"]
.sum()
    )



media_abertura_mensal_periodo = (
abertura_mensal_periodo
.groupby("periodo_pandemia", as_index = False)["abertura_empresas"]
.mean()
.sort_values("abertura_empresas", ascending = False)
.rename(columns = {"abertura_empresas" : "media_mensal_aberturas"})
)

## Segmentos por periodo

Olhei os segmentos em cada periodo para identificar quais grupos mudaram mais ao longo do tempo.


In [ ]:
segmentos_mensal_periodo = (
df
.groupby(["class_segmentos","ano", "mes", "periodo_pandemia"], as_index = False)
["abertura_empresas"]
.sum()
.sort_values("class_segmentos")
)

media_segmentos_periodo = (
segmentos_mensal_periodo
.groupby(["class_segmentos", "periodo_pandemia"], as_index = False)
["abertura_empresas"]
.mean()
.rename(columns = {"abertura_empresas" : "media_mensal_aberturas"})
)

melhores_segmentos =( 
media_segmentos_periodo
.groupby("class_segmentos", as_index = False)
["media_mensal_aberturas"]
.sum()
.sort_values("media_mensal_aberturas", ascending = False)
.head(10)["class_segmentos"]
)

top10 = media_segmentos_periodo [
    media_segmentos_periodo["class_segmentos"].isin(melhores_segmentos)
]

tabela_plot = (top10.pivot(
    index = "class_segmentos",
    columns = "periodo_pandemia",
    values = "media_mensal_aberturas"
)).fillna(0)

## Cidades agrupadas

Calculei a media mensal por cidade agrupada para entender a concentracao regional das aberturas.


In [ ]:
cidades_mensal = (
df
.groupby(["cidade_agrupada", "ano", "mes"],as_index = False)
["abertura_empresas"]
.sum()
)

top_cidades_media_mensal = (
cidades_mensal
.groupby("cidade_agrupada", as_index = False)
["abertura_empresas"]
.mean()
.rename(columns = {"abertura_empresas" : "media_mensal_aberturas"})
.sort_values("media_mensal_aberturas", ascending = False)
)

top_cidades_media_mensal

,cidade_agrupada,media_mensal_aberturas
7,outros,7830.322917
5,SAO PAULO,5424.010417
6,grande_sp,1435.406250
4,RIO DE JANEIRO,1333.687500
0,BELO HORIZONTE,1107.687500
1,CURITIBA,1015.947917
3,PORTO ALEGRE,597.187500
2,FLORIANOPOLIS,356.020833


## Cidades por periodo

Comparei as cidades agrupadas entre periodos para ver se alguma regiao teve uma mudança relevande de acordo com o periodo.


In [142]:
cidades_mensal_periodo = (
df
.groupby(["cidade_agrupada","ano", "mes", "periodo_pandemia"], as_index = False)
["abertura_empresas"]
.sum()
.sort_values("abertura_empresas")
)

media_cidades_mensal_periodo = (
cidades_mensal_periodo
.groupby(["cidade_agrupada", "periodo_pandemia"], as_index = False)
["abertura_empresas"]
.mean()
.rename(columns = {"abertura_empresas" : "media_mensal_aberturas"})
)

top_cidades_periodo =(
media_cidades_mensal_periodo

.pivot(
    index = "cidade_agrupada",
    columns = "periodo_pandemia",
    values = "media_mensal_aberturas"
)

)

print(top_cidades_periodo)


periodo_pandemia  Durante pandemia  Pré pandemia  Pós pandemia
cidade_agrupada                                               
BELO HORIZONTE          942.000000    667.375000   1410.687500
CURITIBA                931.125000    656.416667   1238.125000
FLORIANOPOLIS           298.458333    189.750000    467.937500
PORTO ALEGRE            517.166667    395.583333    738.000000
RIO DE JANEIRO         1310.791667   1029.791667   1497.083333
SAO PAULO              4559.125000   3455.416667   6840.750000
grande_sp              1351.125000   1036.875000   1676.812500
outros                 7355.291667   5495.625000   9235.187500


## Serie mensal total

Criei a serie historica mensal total para enxergar a evolucao geral do mercado ao longo do tempo.


In [ ]:
serie_historica_mensal = (
df
.groupby(["ano", "mes"], as_index = False)["abertura_empresas"]
.sum()
)

serie_historica_mensal["data"] = pd.to_datetime(
    serie_historica_mensal["ano"].astype(str) + "-" +
    serie_historica_mensal["mes"].astype(str) + "-01"
)

serie_historica_mensal = serie_historica_mensal.sort_values("data")

serie_historica_mensal.head()


## Media por cidade

Voltei para a visao por cidade para comparar o volume medio entre as regioes agrupadas.


In [ ]:
media_mensal_cidade = (
cidades_mensal
.groupby("cidade_agrupada", as_index = False)["abertura_empresas"]
.mean()
.rename(columns = {"abertura_empresas" : "media_mensal_aberturas"})
.sort_values("media_mensal_aberturas", ascending = False)
)

media_mensal_cidade


In [ ]:
media_mensal_cidade_periodo = (
cidades_mensal_periodo
.groupby(["cidade_agrupada", "periodo_pandemia"], as_index = False)["abertura_empresas"]
.mean()
.rename(columns = {"abertura_empresas" : "media_mensal_aberturas"})
)

pivot_cidade_periodo = (
media_mensal_cidade_periodo
.pivot(
    index = "cidade_agrupada",
    columns = "periodo_pandemia",
    values = "media_mensal_aberturas"
)
.fillna(0)
)

coluna_pos = [coluna for coluna in pivot_cidade_periodo.columns if coluna.endswith("s pandemia")][0]
pivot_cidade_periodo = pivot_cidade_periodo.sort_values(coluna_pos, ascending = False)

pivot_cidade_periodo


## Participacao por cidade

Calculei a participacao percentual de cada cidade agrupada.


In [ ]:
participacao_cidade = (
df
.groupby("cidade_agrupada", as_index = False)["abertura_empresas"]
.sum()
)

participacao_cidade["participacao_percentual"] = (
    participacao_cidade["abertura_empresas"] / participacao_cidade["abertura_empresas"].sum() * 100
)

participacao_cidade = participacao_cidade.sort_values("participacao_percentual", ascending = False)

participacao_cidade


## Participacao por segmento

Calculei a participacao de cada segmento para entender quais tipos de empresa tem maior peso no mercado.


In [ ]:
participacao_segmento = (
df
.groupby("class_segmentos", as_index = False)["abertura_empresas"]
.sum()
)

participacao_segmento["participacao_percentual"] = (
    participacao_segmento["abertura_empresas"] / participacao_segmento["abertura_empresas"].sum() * 100
)

participacao_segmento = participacao_segmento.sort_values("participacao_percentual", ascending = False)

participacao_segmento.head(10)


## Ultimo mes de 2025

Identifiquei o ultimo mes disponivel de 2025 para saber ate onde vai o historico usado na analise.


In [ ]:
ultimo_mes_2025 = df[df["ano"] == 2025]["mes"].max()
ultimo_mes_2025


In [ ]:
df_periodo_recente = df[
    (df["ano"].isin([2022, 2023, 2024, 2025])) &
    (df["mes"] <= ultimo_mes_2025)
]

crescimento_segmento = (
df_periodo_recente
.groupby(["class_segmentos", "ano"], as_index = False)["abertura_empresas"]
.sum()
.pivot(
    index = "class_segmentos",
    columns = "ano",
    values = "abertura_empresas"
)
.fillna(0)
)

crescimento_segmento = crescimento_segmento[crescimento_segmento[2022] >= 5000]

crescimento_segmento["crescimento_2022_2025_%"] = (
    (crescimento_segmento[2025] / crescimento_segmento[2022] - 1) * 100
)

crescimento_segmento = crescimento_segmento.sort_values("crescimento_2022_2025_%", ascending = False)

crescimento_segmento.head(10)


## Crescimento recente

Calculei o crescimento mais recente por cidade para ver quais regioes estavam acelerando ou desacelerando.


In [ ]:
crescimento_cidade = (
df_periodo_recente
.groupby(["cidade_agrupada", "ano"], as_index = False)["abertura_empresas"]
.sum()
.pivot(
    index = "cidade_agrupada",
    columns = "ano",
    values = "abertura_empresas"
)
.fillna(0)
)

crescimento_cidade["crescimento_2022_2025_%"] = (
    (crescimento_cidade[2025] / crescimento_cidade[2022] - 1) * 100
)

crescimento_cidade = crescimento_cidade.sort_values("crescimento_2022_2025_%", ascending = False)

crescimento_cidade


## Volatilidade por segmento

Analisei a variacao mensal dos segmentos para entender quais grupos oscilam mais ao longo do tempo.


In [ ]:
segmentos_mensal = (
df
.groupby(["class_segmentos", "ano", "mes"], as_index = False)["abertura_empresas"]
.sum()
)

volatilidade_segmento = (
segmentos_mensal
.groupby("class_segmentos")["abertura_empresas"]
.agg(["mean", "std", "sum", "count"])
.rename(columns = {
    "mean" : "media_mensal",
    "std" : "desvio_padrao",
    "sum" : "total_aberturas",
    "count" : "quantidade_meses"
})
)

volatilidade_segmento["coeficiente_variacao"] = (
    volatilidade_segmento["desvio_padrao"] / volatilidade_segmento["media_mensal"]
)

volatilidade_segmento = volatilidade_segmento[volatilidade_segmento["total_aberturas"] >= 5000]
volatilidade_segmento = volatilidade_segmento.sort_values("coeficiente_variacao", ascending = False)

volatilidade_segmento.head(10)


## Volatilidade por cidade

Tambem olhei a volatilidade por cidade agrupada, ja que algumas regioes podem ter comportamento mais estavel que outras.


In [ ]:
volatilidade_cidade = (
cidades_mensal
.groupby("cidade_agrupada")["abertura_empresas"]
.agg(["mean", "std", "sum", "count"])
.rename(columns = {
    "mean" : "media_mensal",
    "std" : "desvio_padrao",
    "sum" : "total_aberturas",
    "count" : "quantidade_meses"
})
)

volatilidade_cidade["coeficiente_variacao"] = (
    volatilidade_cidade["desvio_padrao"] / volatilidade_cidade["media_mensal"]
)

volatilidade_cidade = volatilidade_cidade.sort_values("coeficiente_variacao", ascending = False)

volatilidade_cidade
